In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score,f1_score,classification_report, confusion_matrix
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import seaborn as sns


C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow")

In [3]:
import dagshub
dagshub.init(repo_owner='Aayush10671', repo_name='yt-comment-sentiment-analysis', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as Aayush10671

Initialized MLflow to track repo "Aayush10671/yt-comment-sentiment-analysis"

Repository Aayush10671/yt-comment-sentiment-analysis initialized!

🏃 View run delightful-cow-189 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0/runs/43b77f78b2f941899c6245832d35f73a
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0


In [4]:
df = pd.read_csv("preprocessed_data.csv")
df.shape

(36793, 2)

In [5]:
# Drop rows where clean_comment is missing
df = df.dropna(subset=['clean_comment'])
# Remove rows where the comment is empty after stripping
df = df[df['clean_comment'].str.strip() != '']
# Ensure all values are strings (just in case)
df['clean_comment'] = df['clean_comment'].astype(str)

In [6]:
df = df.dropna(subset=['clean_comment'])
df = df[df['clean_comment'].str.strip() != '']
df['clean_comment'] = df['clean_comment'].astype(str)

In [7]:
mlflow.set_experiment("light_gbm_final")

2026/07/26 01:28:32 INFO mlflow.tracking.fluent: Experiment with name 'light_gbm_final' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/e4ce610bf28f4697820c78c39ea4ef7b', creation_time=1785009514694, experiment_id='8', last_update_time=1785009514694, lifecycle_stage='active', name='light_gbm_final', tags={}, workspace='default'>

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import ADASYN

# -----------------------------
# Split first (avoids leakage)
# -----------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_comment"],
    df["category"],
    test_size=0.2,
    random_state=42,
    stratify=df["category"]
)

# -----------------------------
# TF-IDF
# -----------------------------
vectorizer = TfidfVectorizer(
    ngram_range=(1,3),
    max_features=10000
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# -----------------------------
# ADASYN
# -----------------------------
adasyn = ADASYN(random_state=42)

X_train, y_train = adasyn.fit_resample(X_train, y_train)

In [10]:
df['category'] = df['category'].map({0:0 , 1:1 , -1:2})

df = df.dropna(subset = ['category'])

In [11]:
def log_model(model_name, model, X_train, X_test, y_train, y_test, params, trial_number):

    with mlflow.start_run(run_name=f"Trial_{trial_number}"):

        mlflow.set_tag("model", model_name)
        mlflow.set_tag("sampling", "ADASYN")
        mlflow.set_tag("vectorizer", "TF-IDF")

        mlflow.log_param("ngram_range", (1,3))
        mlflow.log_param("max_features",10000)

        for key,value in params.items():
            mlflow.log_param(key,value)

        model.fit(X_train,y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test,y_pred)

        mlflow.log_metric("accuracy",accuracy)

        report = classification_report(
            y_test,
            y_pred,
            output_dict=True
        )

        for label,metrics in report.items():

            if isinstance(metrics,dict):

                for metric_name,metric_value in metrics.items():

                    mlflow.log_metric(
                        f"{label}_{metric_name}",
                        metric_value
                    )

        mlflow.sklearn.log_model(
            sk_model=model,
            name=f"{model_name}_model"
        )

    return accuracy

In [12]:
import optuna
from lightgbm import LGBMClassifier

def objective(trial):

    params = {

        "n_estimators": trial.suggest_int("n_estimators",100,500),

        "learning_rate": trial.suggest_float("learning_rate",0.01,0.3),

        "max_depth": trial.suggest_int("max_depth",3,15),

        "num_leaves": trial.suggest_int("num_leaves",20,150),

        "min_child_samples": trial.suggest_int("min_child_samples",10,50),

        "subsample": trial.suggest_float("subsample",0.6,1.0),

        "colsample_bytree": trial.suggest_float("colsample_bytree",0.6,1.0),

        "random_state":42,

        "verbosity":-1

    }

    model = LGBMClassifier(**params)

    accuracy = log_model(
        "LightGBM",
        model,
        X_train,
        X_test,
        y_train,
        y_test,
        params,
        trial.number
    )

    return accuracy

In [15]:
def run_optuna_experiment(n_trials=10):

    study = optuna.create_study(direction="maximize")

    study.optimize(objective,n_trials=n_trials)

    print("="*60)
    print("Best Accuracy :",study.best_value)
    print("Best Parameters")
    print(study.best_params)
    print("="*60)

    return study

In [14]:
study = run_optuna_experiment(n_trials=10)

[I 2026-07-26 01:39:19,250] A new study created in memory with name: no-name-0baad328-5bcd-4735-9c0f-dc424b495dd5
C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/26 01:40:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Trial_0 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/8/runs/6c8f059efe4141f89818cda09b0f1d8a
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/8


[I 2026-07-26 01:40:34,660] Trial 0 finished with value: 0.8514932496931679 and parameters: {'n_estimators': 375, 'learning_rate': 0.17589080957638928, 'max_depth': 6, 'num_leaves': 134, 'min_child_samples': 27, 'subsample': 0.9182996474110532, 'colsample_bytree': 0.7737207091539088}. Best is trial 0 with value: 0.8514932496931679.
C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/26 01:42:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Trial_1 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/8/runs/103f00663af84b7cb2ed97b7451cb887
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/8


[I 2026-07-26 01:42:25,053] Trial 1 finished with value: 0.8486294831583254 and parameters: {'n_estimators': 450, 'learning_rate': 0.09075757397958403, 'max_depth': 13, 'num_leaves': 37, 'min_child_samples': 32, 'subsample': 0.8173726615351847, 'colsample_bytree': 0.6288528422931974}. Best is trial 0 with value: 0.8514932496931679.
C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/26 01:44:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run Trial_7 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/8/runs/9f429fcd234b4b899922fa674ea3e37d
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/8


[I 2026-07-26 01:52:37,970] Trial 7 finished with value: 0.8412655120687303 and parameters: {'n_estimators': 147, 'learning_rate': 0.23486274743350527, 'max_depth': 11, 'num_leaves': 108, 'min_child_samples': 36, 'subsample': 0.9427670743597638, 'colsample_bytree': 0.8014896081841693}. Best is trial 6 with value: 0.8520387290331378.
C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/26 01:53:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Trial_8 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/8/runs/1db00996739843088fce828b430170c6
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/8


[I 2026-07-26 01:54:07,253] Trial 8 finished with value: 0.8228555843447429 and parameters: {'n_estimators': 182, 'learning_rate': 0.07674423363063151, 'max_depth': 14, 'num_leaves': 26, 'min_child_samples': 13, 'subsample': 0.9028245634137524, 'colsample_bytree': 0.9525946795683097}. Best is trial 6 with value: 0.8520387290331378.


Best Accuracy : 0.8520387290331378
Best Parameters
{'n_estimators': 259, 'learning_rate': 0.23734071509436078, 'max_depth': 6, 'num_leaves': 70, 'min_child_samples': 13, 'subsample': 0.8100620327135319, 'colsample_bytree': 0.8043874507765093}


In [16]:
best_model = LGBMClassifier(
    **study.best_params,
    random_state=42,
    verbosity=-1
)

with mlflow.start_run(run_name="Best_LightGBM_Model"):

    best_model.fit(X_train, y_train)

    y_pred = best_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    mlflow.log_param("best_params", str(study.best_params))
    mlflow.log_metric("accuracy", accuracy)

    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="Best_LightGBM_Model"
    )

print("Best Parameters:", study.best_params)
print("Best Accuracy:", accuracy)

C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/26 01:57:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Best_LightGBM_Model at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/8/runs/91275015b9ce408ea8d6f44332cd27eb
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/8
Best Parameters: {'n_estimators': 259, 'learning_rate': 0.23734071509436078, 'max_depth': 6, 'num_leaves': 70, 'min_child_samples': 13, 'subsample': 0.8100620327135319, 'colsample_bytree': 0.8043874507765093}
Best Accuracy: 0.8520387290331378
